# Can we fit $\mathcal{H}$ Using DNNs at a Single Kinematic Point Using the KM15/BKM10 Formalism for the Cross-Section **using Experimental Data Only**?

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Native Libraries:

In [ ]:
import os
import sys
import glob
import datetime

### (1.2): Import 3rd-Party Libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots
import numpy as np
import tensorflow as tf
import corner

from scipy.stats import norm
from sklearn.model_selection import train_test_split
import gepard as g
from gepard.fits import th_KM15

from bkm10_lib.core import DifferentialCrossSection
from bkm10_lib.inputs import BKM10Inputs
from bkm10_lib.cff_inputs import CFFInputs

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: tensorflow version: {tf.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")
print(f"[INFO]: corner version: {corner.__version__}")

### (1.4): Customizing Plotting Settings:

In [ ]:
plt.style.use('science')

## (2): Data Formatting/Collection Settings:

In [ ]:
VERSION_NUMBER = 1
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

## (3): Data Loading and Analysis:

### (3.1): Loading in `gepard`'s Datasets:

**Remark:** `g.dset` is a dictionary of `gepard` `DataSet` classes.
```python
g.dset[73]
> DataSet with 39 points
```

**Remark:** Some of the `DataPoint`s in the `DataSet`s *do not have all the right kinematics!* See below for this:

In [ ]:
try:
    print(f"[INFO]: k = {g.dset[47][0].in1energy}")
    print(f"[INFO]: xb = {g.dset[47][0].xB}")
    print(f"[INFO]: t = {g.dset[47][0].t}")
    print(f"[INFO]: Q^2 = {g.dset[47][0].Q2}")
except AttributeError:
    print(f"[ERROR]: Missing crucial kinematic setting information!")

### (3.2): Determining which `gepard` `DataSet`s have all of the right Kinematic Variables:

[NOTE]: We are looking for $k$, $x_{\textrm{B}}$, $t$, and $Q^{2}$.

In [ ]:
required_attributes = ['xB', 't', 'Q2', 'in1energy', 'observable', 'val', 'err']
valid_datasets = []

for dataset_index, dataset in g.dset.items():

    # check the first datapoint in the DataSet for contents!
    first_gepard_datapoint = dataset[0] if len(dataset) > 0 else None
    
    if first_gepard_datapoint and all(hasattr(first_gepard_datapoint, kinematic_attribute) for kinematic_attribute in required_attributes):
        valid_datasets.append(dataset_index)

print(f"[INFO]: Valid dataset indices are:\n{sorted(valid_datasets)}")
print(f"[INFO]: Length of valid datasets = {len(valid_datasets)}")
print(f"[INFO]: Compare length of *all* dataset = {len(g.dset)}")
print(f"[INFO]: Total invalid (according to our criteria) datasets = {len(g.dset) - len(valid_datasets)}")

According to [gepard documentation](https://gepard.phy.hr/docs/observables.htmlhttps://gepard.phy.hr/docs/observables.html), here are the relevant observables for our analysis

1. `XGAMMA` = basically DVCS cross-section
2. `XUU` = beam-averaged cross-section
1. `AC` = beam charge asymmetry
1. `ALU` = beam spin asymmetry
2. `AUL` = longitudinally-polarized target asymmetry
3. `AUT` = transversely-polarized target asymmetry

We need to figure out which `DataSet`s have the observables we're looking for!

### (3.3): Paritioning Datasets into Dictionary of Desired Observables

In [ ]:
target_observables = {'XGAMMA', 'XUU', 'AC', 'ALU', 'AUL'}

# dictionary of string-to-list key-value pairs:
desired_observable_dictionary = {observable: [] for observable in target_observables}

for dataset_index in valid_datasets:
    if not hasattr(g.dset[dataset_index][0], "observable"):
        print(f"[WARN]: Dataset {g.dset[dataset_index]} had datapoint without observable property...")
        continue
    
    observable_name = g.dset[dataset_index][0].observable
    
    if observable_name in desired_observable_dictionary:
        desired_observable_dictionary[observable_name].append(dataset_index)

for name, indices in desired_observable_dictionary.items():
    print(f"{name}: {indices}")

[NOTE]: From the brief code snippet above, we see that the *majority* of `gepard` `DataSet`s involve `"XUU"` and `"ALU"`, which justifies a simultaneous fit in the observables $d^{4}\sigma^{UU}$ and $\textrm{BSA}\left( 0 \right)$.

In [ ]:
rows_for_experimentally_derived_pseudodata = []
rows_for_experimental_data_only = []
global_set_counter = 1

for experiment_id in desired_observable_dictionary['XUU']:

    dataset = g.dset[experiment_id]
    print(f"[INFO]: Now iterating on Experiment {dataset.collaboration} ({dataset.year})")

    kinematic_set_number = 1 
    
    for datapoint in dataset:
        if not hasattr(datapoint, "observable"):
            print(f"[WARN]: Datapoint for Experiment {g.dset[experiment_id].collaboration} ({g.dset[experiment_id].year}) has no observable...")
        else:
            assert datapoint.observable == "XUU", "[ASSERT]: Current datapoint observable not matching!"

        if all(hasattr(datapoint, attr) for attr in ["in1energy", "xB", "Q2", "t","phi"]):
            
            # extact the CFFs using km15
            km15_real_h = th_KM15.ReH(datapoint)
            km15_imag_h = th_KM15.ImH(datapoint)
            km15_real_e = th_KM15.ReE(datapoint)
            km15_imag_e = th_KM15.ImE(datapoint)
            km15_real_ht = th_KM15.ReHt(datapoint)
            km15_imag_ht = th_KM15.ImHt(datapoint)
            km15_real_et = th_KM15.ReEt(datapoint)
            km15_imag_et = th_KM15.ImEt(datapoint)

            km15_bkm10_cross_section = DifferentialCrossSection(
                configuration = {
                    "kinematics": BKM10Inputs(
                        lab_kinematics_k = datapoint.in1energy,
                        squared_Q_momentum_transfer = datapoint.Q2,
                        x_Bjorken = datapoint.xB,
                        squared_hadronic_momentum_transfer_t = datapoint.t),
                    "cff_inputs": CFFInputs(
                        compton_form_factor_h = complex(km15_real_h, km15_imag_h),
                        compton_form_factor_h_tilde = complex(km15_real_ht, km15_imag_ht),
                        compton_form_factor_e = complex(km15_real_e, km15_imag_e),
                        compton_form_factor_e_tilde = complex(km15_real_et, km15_imag_et)),
                    "using_ww": True
                },
                verbose = False,
                debugging = False)
            
            unpolarized_cross_section = km15_bkm10_cross_section.compute_cross_section(
                datapoint.phi,
                lepton_helicity = 0.0,
                target_polarization = 0.0).real
            
            bkm10_bsa_km15 = km15_bkm10_cross_section.compute_bsa(
                datapoint.phi, 
                target_polarization = 0.0).real

            experimentally_derived_pseudodata = {
                "set": global_set_counter,
                "k": datapoint.in1energy,
                "q_squared": datapoint.Q2,
                "x_b": datapoint.xB,
                "t": datapoint.t,
                "phi": datapoint.phi,
                "unp_beam_unp_target_xsec": unpolarized_cross_section[0], # [0] index needed because datapoint.phi is *not* an array...
                "unp_beam_unp_target_xsec_err": 0.0,
                "unp_beam_unp_target_xsec_errstat": 0.0,
                "unp_beam_unp_target_xsec_errsyst": 0.0,
                "unp_target_bsa": bkm10_bsa_km15[0], # [0] index needed because datapoint.phi is *not* an array...
                "unp_target_bsa_err": 0.0,
                "unp_target_bsa_errstat": 0.0,
                "unp_target_bsa_errsyst": 0.0,
                "Re[H]": km15_real_h, "Im[H]": km15_imag_h,
                "Re[E]": km15_real_e, "Im[E]": km15_imag_e,
                "Re[Ht]": km15_real_ht, "Im[Ht]": km15_imag_ht,
                "Re[Et]": km15_real_et, "Im[Et]": km15_imag_et,
                "experiment_year": f"{dataset.collaboration}_{dataset.year}",
                "flag": "unknown"
            }

            experimental_data_point = {
                "set": global_set_counter,
                "k": datapoint.in1energy,
                "q_squared": datapoint.Q2,
                "x_b": datapoint.xB,
                "t": datapoint.t,
                "phi": datapoint.phi,
                "unp_beam_unp_target_xsec": datapoint.val,
                "unp_beam_unp_target_xsec_err": datapoint.err,
                "unp_beam_unp_target_xsec_errstat": datapoint.errstat,
                "unp_beam_unp_target_xsec_errsyst": datapoint.errsyst,
                "unp_target_bsa": 0.0,
                "unp_target_bsa_err": 0.0,
                "unp_target_bsa_errstat": 0.0,
                "unp_target_bsa_errsyst": 0.0,
                "Re[H]": km15_real_h, "Im[H]": km15_imag_h,
                "Re[E]": km15_real_e, "Im[E]": km15_imag_e,
                "Re[Ht]": km15_real_ht, "Im[Ht]": km15_imag_ht,
                "Re[Et]": km15_real_et, "Im[Et]": km15_imag_et,
                "coordinate_frame": datapoint.frame,
                "experiment_year": f"{dataset.collaboration}_{dataset.year}",
                "flag": "unknown"
            }

            rows_for_experimentally_derived_pseudodata.append(experimentally_derived_pseudodata)
            rows_for_experimental_data_only.append(experimental_data_point)

            del experimentally_derived_pseudodata
            del experimental_data_point
            del km15_bkm10_cross_section
        else:
            print(f"[WARN]: Missing kinematics in {dataset.collaboration}")

    global_set_counter += 1
    del dataset


df_expermentally_derived = pd.DataFrame(rows_for_experimentally_derived_pseudodata)
df_expermental_data = pd.DataFrame(rows_for_experimental_data_only)

print(f"[INFO]: Total number of rows in exp-derived DF: {len(df_expermentally_derived)}")
print(f"[INFO]: Total number of rows in exp-only DF: {len(df_expermental_data)}")

print(f"[INFO]: We are now reassigning the kinematic set index such that unique (k, x_b, q_squared, t) values each get their own set index.")

# previous loop did *not* correctly assign kinematic settings; we now remedy the issue using ngroup()... thanks to Google Gemini!
# [NOTE]: WE ROUND BY 4 BELOW: THAT IS EFFECTIVELY BINNING THE KINEMATIC SPACE!
df_expermentally_derived['set'] = df_expermentally_derived.round(4).groupby(['k', 'x_b', 't', 'q_squared'], sort = False).ngroup() + 1
df_expermental_data['set'] = df_expermental_data.round(4).groupby(['k', 'x_b', 't', 'q_squared'], sort = False).ngroup() + 1
print(f"[INFO]: Dataframe kinematic bins now should be uniquely indexed by kinematic set index! Checking...")

unique_sets_exp_derived = df_expermentally_derived['set'].nunique()
unique_sets_exp_data = df_expermental_data['set'].nunique()
print(f"[INFO]: Total Datapoints in exp-derived DF: {unique_sets_exp_derived}")
print(f"[INFO]: Total Datapoints in exp-only DF: {unique_sets_exp_data}")

df_expermentally_derived.to_csv(path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/main_pseudodata_file_v{MAJOR_MINOR_NUMBER}.csv")
df_expermental_data.to_csv(path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/experimental_data_v{MAJOR_MINOR_NUMBER}.csv")

### (3.4): Determining the Unique Kinematic Settings for Plotting:

In [ ]:
# thanks Google Gemini!
kinematic_summary_dataframe = df_expermental_data.groupby(['experiment_year', 'set']).agg({
    'x_b': 'first',
    't': 'first',
    'q_squared': 'first'
}).reset_index()

print(f"[INFO]: Extracted {len(kinematic_summary_dataframe)} unique experiment-kinematic pairs.")
print(f"[INFO]: The number of unique kinematic points should be the same as the number we just found. Is it? {len(kinematic_summary_dataframe) == unique_sets_exp_data}")

#### (X.Y): Scatterplot of All Experimental Datapoints:

In [ ]:
all_uuxsec_experiments_figure = plt.figure(figsize = (9, 7))
all_uuxsec_experiments_axis = all_uuxsec_experiments_figure.add_subplot(1, 1, 1, projection = "3d")

axis_elevation = all_uuxsec_experiments_axis.elev # extract eleveation param
axis_azimuthal = all_uuxsec_experiments_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
all_uuxsec_experiments_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = all_uuxsec_experiments_axis.transAxes)
all_uuxsec_experiments_axis.text2D(
    0.01, 0.00, 
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = all_uuxsec_experiments_axis.transAxes)

for experiment in kinematic_summary_dataframe['experiment_year'].unique():
    experiment_kinematic_set_data = kinematic_summary_dataframe[kinematic_summary_dataframe['experiment_year'] == experiment]
    all_uuxsec_experiments_axis.scatter(
        experiment_kinematic_set_data['x_b'], experiment_kinematic_set_data['q_squared'], experiment_kinematic_set_data['t'],
        label = experiment)

all_uuxsec_experiments_axis.set_title(rf"Experimental Kinematic Settings Space for $d^{{4}}\sigma^{{UU}}$", fontsize = 18)
all_uuxsec_experiments_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 14)
all_uuxsec_experiments_axis.set_ylabel(r"$Q^{2}$", fontsize = 14)
all_uuxsec_experiments_axis.set_zlabel(r"$-t$", fontsize = 14)
all_uuxsec_experiments_axis.legend()

all_uuxsec_experiments_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/experimental_kinematic_space_for_unp_beam_unp_target_xsec_v{MAJOR_MINOR_NUMBER}.png")
all_uuxsec_experiments_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/experimental_kinematic_space_for_unp_beam_unp_target_xsec_v{MAJOR_MINOR_NUMBER}.eps")
plt.close(all_uuxsec_experiments_figure)

#### (X.Y): Scatterplot of Experimental Datapoints Separately:

In [ ]:
for experiment in kinematic_summary_dataframe['experiment_year'].unique():
    single_experiment_figure = plt.figure(figsize = (9, 7))
    single_experiment_axis = single_experiment_figure.add_subplot(1, 1, 1, projection = "3d")

    experimental_subset = kinematic_summary_dataframe[kinematic_summary_dataframe['experiment_year'] == experiment]
    single_experiment_axis.scatter(
        experimental_subset['x_b'], experimental_subset['q_squared'], experimental_subset['t'],
        label = experiment, color = "grey")

    single_experiment_axis.set_title(rf"{experiment} Kinematic Coverage for $d^{{4}}\sigma^{{UU}}$", fontsize = 18)
    single_experiment_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 14)
    single_experiment_axis.set_ylabel(r"$Q^{2}$", fontsize = 14)
    single_experiment_axis.set_zlabel(r"$-t$", fontsize = 14)
    single_experiment_axis.legend()

    single_experiment_axis.set_xlim(left = 0.1 - 0.05, right = 0.6 + 0.05)
    single_experiment_axis.set_ylim(bottom = 1.0 - 0.1, top = 9.0 + 0.1)
    single_experiment_axis.set_zlim(-1.4 - 0.1, -0.2 + 0.1)

    single_experiment_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/{experiment}_kinematic_space_unp_beam_unp_target_xsec_v{MAJOR_MINOR_NUMBER}.png")
    single_experiment_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/{experiment}_kinematic_space_unp_beam_unp_target_xsec_v{MAJOR_MINOR_NUMBER}.eps")
    plt.close(single_experiment_figure)

    del single_experiment_figure
    del single_experiment_axis

In [ ]:
xb_data = [p.xB for p in g.dset[100]]
q_squared_data = [p.Q2 for p in g.dset[100]]
t_data = [p.t for p in g.dset[100]]
observable = g.dset[100].observable

# unpolarized beam | unpolarized target
if observable == "XUU":
    label_plot = r"$d^{4}\sigma^{UU}$"
    column = "unp_beam_unp_target_xsec"
# BSA | unpolarized target:
elif observable == "ALU":
    label_plot = r"$\text{BSA} \left( \Lambda = 0 \right)$"
    column = "unp_target_bsa"
else:
    print(f"[WARN]: Observable {observable} unrecognized.")
    label_plot = 'unknown'
    column = 'unknown'

In [ ]:
for kinematic_set_id, experiment in df_expermental_data.groupby('set'):

    os.makedirs(name = )

    exp_year = experiment['experiment_year'].iloc[0]
    k_value = experiment['k'].iloc[0]
    xb_value = experiment['x_b'].iloc[0]
    t_value = experiment['t'].iloc[0]
    q2_value = experiment['q_squared'].iloc[0]
    
    uu_xsec_vs_phi_figure = plt.figure(figsize = (9, 6))
    uu_xsec_vs_phi_axis = uu_xsec_vs_phi_figure.add_subplot(1, 1, 1)
    if experiment["coordinate_frame"].iloc[0] == "BMK":
        uu_xsec_vs_phi_axis.errorbar(
            x = experiment['phi'], y = experiment['unp_beam_unp_target_xsec'], 
            yerr = experiment['unp_beam_unp_target_xsec_err'], 
            fmt = 'o', capsize = 3)
    elif experiment["coordinate_frame"].iloc[0] == "Trento":
        uu_xsec_vs_phi_axis.errorbar(
            x = experiment['phi'] + np.pi, y = experiment['unp_beam_unp_target_xsec'], 
            yerr = experiment['unp_beam_unp_target_xsec_err'], 
            fmt = 'o', capsize = 3)

    uu_xsec_vs_phi_axis.set_title(rf"{exp_year} $d^{{4}}\sigma^{{UU}}$ vs. $\phi$, $k = {k_value} $, $x_\textrm{{B}} = {xb_value}$, $t = {t_value}$, $Q^{{2}} = {q2_value}$")
    uu_xsec_vs_phi_axis.set_xlabel(r"$\phi$ (radians)")
    uu_xsec_vs_phi_axis.set_ylabel(r"$d^{{4}}\sigma^{{UU}}$")
    uu_xsec_vs_phi_axis.grid(True, linestyle = '--', alpha = 0.6)
    uu_xsec_vs_phi_axis.legend()
     
    uu_xsec_vs_phi_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/{exp_year}_setid_{kinematic_set_id}_uu_xsec_vs_phi_v{MAJOR_MINOR_NUMBER}.png")
    plt.close(uu_xsec_vs_phi_figure)

    del uu_xsec_vs_phi_figure
    del uu_xsec_vs_phi_axis